# FDA Drug Adverse Event Signal Detection & Pharmacovigilance Analysis
## Identifying Safety Signals Across High-Volume Pharmaceutical Products

### Project Overview
This analysis applies pharmacovigilance methodology to the FDA Adverse Event 
Reporting System (FAERS) to detect drug safety signals across five high-volume 
pharmaceutical products. Using the Reporting Odds Ratio (ROR), the industry 
standard signal detection method used by regulatory data teams at major 
pharmaceutical organisations, this project identifies statistically elevated 
adverse event patterns, seasonal trends, demographic risk profiles and 
longitudinal performance across Aspirin, Ibuprofen, Paracetamol, Metformin 
and Atorvastatin.

### Objectives
- Detect adverse event safety signals using Reporting Odds Ratio methodology
- Analyse monthly and quarterly adverse event volume trends
- Compare serious event and death rates across five pharmaceutical products
- Identify demographic risk profiles by age group and sex
- Measure seasonal variation in adverse event reporting
- Track longitudinal trends in hospitalisation and mortality rates
- Present findings in an interactive Power BI dashboard

### Tools and Technologies
| Tool | Purpose |
|---|---|
| Python 3.12 | Core programming language |
| pandas | Data manipulation and transformation |
| numpy | Statistical calculations |
| matplotlib | Data visualisation |
| seaborn | Statistical visualisation |
| scipy | Signal detection statistics |
| PostgreSQL 16 | Database storage and SQL analysis |
| SQLAlchemy | Python to PostgreSQL connection |
| psycopg2 | PostgreSQL database adapter |
| Jupyter Lab | Interactive analysis environment |
| Power BI Desktop | Interactive dashboard |
| openFDA API | Primary data source |
| Git | Version control |

### Data Source
| | |
|---|---|
| **Publisher** | U.S. Food and Drug Administration |
| **Dataset** | FDA Adverse Event Reporting System (FAERS) |
| **Access** | openFDA API open.fda.gov/apis/drug/event |
| **Coverage** | Post-market adverse event reports 2004 to present |
| **Frequency** | Quarterly updates |
| **Licence** | Public domain CC0 |

*Analysis conducted by Kingsley Eboh | GitHub: Kingsley-Eboh*

In [1]:
# Step 1: Import Libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

All libraries loaded successfully
pandas: 3.0.3
numpy: 2.4.6


### Section 1: Data Acquisition
Adverse event data was retrieved from the FDA Adverse Event Reporting System 
(FAERS) via the openFDA API for five high volume pharmaceutical products: 
Aspirin, Ibuprofen, Paracetamol, Metformin and Atorvastatin. The data pull 
was restricted to reports received between January 2022 and December 2025 
to ensure the analysis reflected current post-market safety surveillance 
activity.

In [2]:

# Section 1: Data Acquisition
import time

base_url = "https://api.fda.gov/drug/event.json"
api_key = "ADD_YOUR_API_KEY_HERE"

drugs = [
    "aspirin",
    "ibuprofen",
    "paracetamol",
    "acetaminophen",
    "metformin",
    "atorvastatin"
]

def pull_fda_data(drug_name, limit=1000):
    params = {
        "search": f"patient.drug.medicinalproduct:{drug_name} AND receivedate:[20220101 TO 20251231]",
        "limit": limit,
        "api_key": api_key
    }
    try:
        response = requests.get(base_url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            print(f"{drug_name}: {data['meta']['results']['total']:,} total records available")
            return data['results']
        else:
            print(f"{drug_name}: Error {response.status_code}")
            return []
    except Exception as e:
        print(f"{drug_name}: Failed - {str(e)}")
        return []

# Pull data for all five drugs with delay between each
all_data = {}
for drug in drugs:
    all_data[drug] = pull_fda_data(drug)
    time.sleep(2)
    
print("\nData pull complete")

aspirin: 130,786 total records available
ibuprofen: 59,612 total records available
paracetamol: 1,182 total records available
acetaminophen: 190,505 total records available
metformin: 103,178 total records available
atorvastatin: 118,400 total records available

Data pull complete


### Section 2: Data Parsing and Structuring
Raw JSON responses from the openFDA API were parsed into a structured tabular format. Key fields were extracted from nested JSON including patient demographics, reaction terms, seriousness indicators and report metadata. A master DataFrame was constructed combining all five drugs into a single unified dataset ready for quality assessment and cleaning.

In [3]:
# Section 2: Data Parsing and Structuring
def parse_records(records, drug_name):
    parsed = []
    for report in records:
        try:
            record = {
                'drug_name': drug_name,
                'report_id': report.get('safetyreportid'),
                'serious': report.get('serious'),
                'seriousnessdeath': report.get('seriousnessdeath', '0'),
                'seriousnesshospitalisation': report.get('seriousnesshospitalization', '0'),
                'seriousnessdisabling': report.get('seriousnessdisabling', '0'),
                'seriousnesslifethreatening': report.get('seriousnesslifethreatening', '0'),
                'receive_date': report.get('receivedate'),
                'reporter_country': report.get('primarysourcecountry'),
                'patient_age': report.get('patient', {}).get('patientonsetage'),
                'patient_age_unit': report.get('patient', {}).get('patientonsetageunit'),
                'patient_sex': report.get('patient', {}).get('patientsex'),
                'patient_weight': report.get('patient', {}).get('patientweight'),
                'reactions': [
                    r.get('reactionmeddrapt') 
                    for r in report.get('patient', {}).get('reaction', [])
                ],
                'reaction_count': len(report.get('patient', {}).get('reaction', [])),
                'drug_count': len(report.get('patient', {}).get('drug', []))
            }
            parsed.append(record)
        except Exception as e:
            pass
    return parsed

# Parse all drugs
all_records = []
for drug, records in all_data.items():
    parsed = parse_records(records, drug)
    all_records.extend(parsed)
    print(f"{drug}: {len(parsed)} records parsed")

# Create master DataFrame
df = pd.DataFrame(all_records)

# Merge paracetamol and acetaminophen as one drug
df['drug_name'] = df['drug_name'].str.capitalize()
df['drug_name'] = df['drug_name'].replace('Acetaminophen', 'Paracetamol')

print(f"\nTotal records in DataFrame: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nRecords per drug after merging:")
print(df['drug_name'].value_counts())

aspirin: 1000 records parsed
ibuprofen: 1000 records parsed
paracetamol: 1000 records parsed
acetaminophen: 1000 records parsed
metformin: 1000 records parsed
atorvastatin: 1000 records parsed

Total records in DataFrame: 6,000
Columns: ['drug_name', 'report_id', 'serious', 'seriousnessdeath', 'seriousnesshospitalisation', 'seriousnessdisabling', 'seriousnesslifethreatening', 'receive_date', 'reporter_country', 'patient_age', 'patient_age_unit', 'patient_sex', 'patient_weight', 'reactions', 'reaction_count', 'drug_count']

DataFrame shape: (6000, 16)

Records per drug after merging:
drug_name
Paracetamol     2000
Aspirin         1000
Ibuprofen       1000
Metformin       1000
Atorvastatin    1000
Name: count, dtype: int64


### Section 3: Data Quality Assessment
The dataset was assessed for missing values, data types and distribution across key fields. This step identified data completeness issues and informed the cleaning strategy applied in the following section.

In [4]:
# Section 3: Data Quality Assessment
print("=== DATA QUALITY ASSESSMENT ===\n")

# Shape
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}\n")

# Missing values
print("Missing values per column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_pct
})
print(missing_df)

# Drug distribution
print(f"\nRecords per drug:")
print(df['drug_name'].value_counts())

# Serious events
print(f"\nSerious events distribution:")
print(df['serious'].value_counts())

=== DATA QUALITY ASSESSMENT ===

Total records: 6,000
Total columns: 16

Missing values per column:
                            Missing Count  Missing Percentage
drug_name                               0                0.00
report_id                               0                0.00
serious                                 0                0.00
seriousnessdeath                        0                0.00
seriousnesshospitalisation              0                0.00
seriousnessdisabling                    0                0.00
seriousnesslifethreatening              0                0.00
receive_date                            0                0.00
reporter_country                       21                0.35
patient_age                          2384               39.73
patient_age_unit                     2384               39.73
patient_sex                           472                7.87
patient_weight                       4018               66.97
reactions                       

### Section 4: Data Cleaning
The dataset was cleaned to ensure consistency and accuracy across all fields. 
Date fields were converted to datetime format. Seriousness indicators were 
mapped to human readable labels. Patient age was converted to a numeric format 
and age groups were assigned. Patient sex codes were mapped to descriptive 
labels. Missing values were handled appropriately for each field. Records with 
missing critical fields were retained where possible to preserve data volume 
for analysis.

In [5]:
# Section 4: Data Cleaning

# Convert receive_date to datetime
df['receive_date'] = pd.to_datetime(df['receive_date'], format='%Y%m%d', errors='coerce')

# Extract year, month and quarter
df['year'] = df['receive_date'].dt.year
df['month'] = df['receive_date'].dt.month
df['quarter'] = df['receive_date'].dt.quarter
df['year_month'] = df['receive_date'].dt.to_period('M')

# Map serious field
df['serious_label'] = df['serious'].map({'1': 'Serious', '2': 'Non-Serious'})

# Map seriousness indicators
df['seriousnessdeath'] = df['seriousnessdeath'].fillna('0')
df['seriousnesshospitalisation'] = df['seriousnesshospitalisation'].fillna('0')
df['seriousnessdisabling'] = df['seriousnessdisabling'].fillna('0')
df['seriousnesslifethreatening'] = df['seriousnesslifethreatening'].fillna('0')

# Map patient sex
df['sex_label'] = df['patient_sex'].map({
    '1': 'Male',
    '2': 'Female'
}).fillna('Unknown')

# Convert patient age to numeric
df['patient_age'] = pd.to_numeric(df['patient_age'], errors='coerce')

# Assign age groups
def assign_age_group(age):
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Under 18'
    elif age < 45:
        return '18 to 44'
    elif age < 65:
        return '45 to 64'
    elif age < 85:
        return '65 to 84'
    else:
        return '85 and over'

df['age_group'] = df['patient_age'].apply(assign_age_group)

# Fill missing reporter country
df['reporter_country'] = df['reporter_country'].fillna('Unknown')

# Capitalise drug names
df['drug_name'] = df['drug_name'].str.capitalize()

print("=== DATA CLEANING COMPLETE ===\n")
print(f"DataFrame shape: {df.shape}")
print(f"\nDate range: {df['receive_date'].min()} to {df['receive_date'].max()}")
print(f"\nSex distribution:")
print(df['sex_label'].value_counts())
print(f"\nAge group distribution:")
print(df['age_group'].value_counts())
print(f"\nSerious events:")
print(df['serious_label'].value_counts())

=== DATA CLEANING COMPLETE ===

DataFrame shape: (6000, 23)

Date range: 2022-01-01 00:00:00 to 2025-04-14 00:00:00

Sex distribution:
sex_label
Female     2868
Male       2660
Unknown     472
Name: count, dtype: int64

Age group distribution:
age_group
Unknown        2384
65 to 84       1534
45 to 64       1115
18 to 44        587
85 and over     270
Under 18        110
Name: count, dtype: int64

Serious events:
serious_label
Serious        4869
Non-Serious    1131
Name: count, dtype: int64
